In [10]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler
import random
from datetime import datetime

In [11]:
TRAIN_FILE_PATH = 'Groceries data train.csv'
TEST_FILE_PATH = 'Groceries data test.csv' # Loaded but not used for evaluation in this script
N_RECOMMENDATIONS = 10 # Number of recommendations to generate

In [13]:
# Load the training data
df_train = pd.read_csv(TRAIN_FILE_PATH)
# Load the test data (optional, for potential future evaluation)
df_test = pd.read_csv(TEST_FILE_PATH)

# Convert 'Date' column to datetime objects
df_train['Date'] = pd.to_datetime(df_train['Date'], format='%d/%m/%Y')
df_test['Date'] = pd.to_datetime(df_test['Date'], format='%d/%m/%Y')

print("Training data loaded successfully:")
print(df_train.head())
print(f"\nTraining data shape: {df_train.shape}")
# print("\nTest data loaded successfully:")
# print(df_test.head())
# print(f"\nTest data shape: {df_test.shape}")

Training data loaded successfully:
   User_id       Date itemDescription  year  month  day  day_of_week
0     2351 2014-01-01         cleaner  2014      1    1            2
1     2226 2014-01-01         sausage  2014      1    1            2
2     1922 2014-01-01  tropical fruit  2014      1    1            2
3     2943 2014-01-01      whole milk  2014      1    1            2
4     1249 2014-01-01    citrus fruit  2014      1    1            2

Training data shape: (19382, 7)


In [16]:
max_date = df_train['Date'].max()

Timestamp('2015-01-20 00:00:00')

In [19]:
# Calculate recency score for each interaction
# Score = 1 / (days difference from max_date + 1)
# Adding 1 to avoid division by zero if purchase is on the max_date
df_train['Recency_Score'] = df_train['Date'].apply(
    lambda date: 1.0 / ((max_date - date).days + 1)
)

In [22]:
# Create the utility matrix: User x Item with Recency_Score as values
# We group by user and item and take the MAX recency score
# (in case a user bought the same item multiple times, we consider the most recent purchase)
utility_matrix_df = df_train.groupby(['User_id', 'itemDescription'])['Recency_Score'].max().unstack(fill_value=0)


print("Utility matrix created successfully.")
print(f"Shape of utility matrix: {utility_matrix_df.shape}")
utility_matrix_df

Utility matrix created successfully.
Shape of utility matrix: (3493, 167)


itemDescription,Instant food products,UHT-milk,abrasive cleaner,artif. sweetener,baby cosmetics,bags,baking powder,bathroom cleaner,beef,berries,...,turkey,vinegar,waffles,whipped/sour cream,whisky,white bread,white wine,whole milk,yogurt,zwieback
User_id,,,,,,,,,,,,,,,,,,,,,
1000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.004739,0.0,0.0
1001,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.025000,0.0,0.0
1002,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.003704,0.0,0.0
1003,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0
1004,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.010000,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4993,0.0,0.0,0.006289,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0
4995,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0
4997,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.003774,0.0,0.0
